## Start a Spark + Delta session in the notebook
Layering the pipeline (bronze → silver → gold) instead of a single cleaning script is a common data engineering practice, for a few reasons:
- *Auditability* — the raw data is preserved at every stage, so each checkpoint can be inspected independently.
- *Reproducibility* — if one stage breaks, it can be fixed and rerun on its own, without redoing the whole pipeline.

Technical details:
- `pyspark.sql` is the module for working with structured data in Spark.
- `SparkSession` is the entry point to programming Spark with the Dataset and DataFrame API.
- The two `.config(...)` lines are what turn on Delta Lake specifically. Without them, Spark would just read/write plain files (CSV, Parquet, etc.) with no transaction log — the ACID/versioning behavior we're building toward on Day 3 comes from these two lines being set now.

In [1]:
from pyspark.sql import SparkSession               
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder              # boot up the Spark engine
    .appName("movielens-day1")
    .master("local[2]")               # running locally on 2 cores, no real cluster involved
    # turning on Delta Lake support in Spark
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate() # makes sure Spark actually downloads/loads the Delta code
spark.sparkContext.setLogLevel("WARN") # reduces the amount of log output to the console

print("Spark version:", spark.version)

Spark version: 3.5.8


In [2]:
from pathlib import Path

# the notebook runs from notebooks/, so the project root is one level up.
# using a relative path (instead of hardcoding "C:/Users/<name>/...") means
# this notebook works on any machine that clones the repo, not just this one.
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (PROJECT_ROOT / "dataset" / "ml-latest-small").as_posix()
BRONZE_DIR = (PROJECT_ROOT / "bronze").as_posix()

print("Data dir:", DATA_DIR)
print("Bronze dir:", BRONZE_DIR)

Data dir: c:/Users/Manar Albader/OneDrive/Desktop/MovieLens/dataset/ml-latest-small
Bronze dir: c:/Users/Manar Albader/OneDrive/Desktop/MovieLens/bronze


## Load the dataset into Spark, look at the inferred schema
- `inferSchema=True` — tells Spark to scan through the file and guess a type for each column (integer, double, string...), rather than treating everything as text. This is a genuinely useful/lazy trick for exploration, but it's a guess, not a guarantee — worth watching for anything that looks wrong.
- `printSchema()` — prints each column's name and the type Spark landed on.

In [3]:
ratings_raw = spark.read.csv(f"{DATA_DIR}/ratings.csv", header=True, inferSchema=True)
movies_raw  = spark.read.csv(f"{DATA_DIR}/movies.csv",  header=True, inferSchema=True)
tags_raw    = spark.read.csv(f"{DATA_DIR}/tags.csv",    header=True, inferSchema=True)
links_raw   = spark.read.csv(f"{DATA_DIR}/links.csv",   header=True, inferSchema=True)

# "name" is just a string to identify the DataFrame, "df" is the actual DataFrame object
for name, df in [("ratings", ratings_raw), ("movies", movies_raw), ("tags", tags_raw), ("links", links_raw)]:
    print(f"--- {name} ---")
    df.printSchema()

--- ratings ---
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

--- movies ---
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

--- tags ---
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)

--- links ---
root
 |-- movieId: integer (nullable = true)
 |-- imdbId: integer (nullable = true)
 |-- tmdbId: integer (nullable = true)



`nullable` just means: is this column allowed to have an empty/missing value (a "null") in this table's schema. Spark defaults everything read from CSV to `nullable = true` since it can't guarantee a column is always filled

## Profile each table
Row counts, key uniqueness, missing values

For each column, `df.filter(df[col].isNull()).count()` counts how many rows have nothing in that column, then we print it as both a raw count and a percentage.

In [4]:
# a function to profile a DataFrame, printing the number of rows and the number of missing values for each column
# it takes name (a string) and df (a DataFrame) as arguments
def profile(name, df):
    total = df.count() # count the number of rows in the DataFrame
    print(f"--- {name}: {total} rows ---")
    for col in df.columns:
        nulls = df.filter(df[col].isNull()).count()
        print(f"  {col}: {nulls} missing ({nulls / total:.2%})")

# a loop to profile each of the DataFrames, using the profile function defined above
for name, df in [("ratings", ratings_raw), ("movies", movies_raw), ("tags", tags_raw), ("links", links_raw)]:
    profile(name, df)

--- ratings: 100836 rows ---
  userId: 0 missing (0.00%)
  movieId: 0 missing (0.00%)
  rating: 0 missing (0.00%)
  timestamp: 0 missing (0.00%)
--- movies: 9742 rows ---
  movieId: 0 missing (0.00%)
  title: 0 missing (0.00%)
  genres: 0 missing (0.00%)
--- tags: 3683 rows ---
  userId: 0 missing (0.00%)
  movieId: 0 missing (0.00%)
  tag: 0 missing (0.00%)
  timestamp: 0 missing (0.00%)
--- links: 9742 rows ---
  movieId: 0 missing (0.00%)
  imdbId: 0 missing (0.00%)
  tmdbId: 8 missing (0.08%)


In [5]:
def check_unique(name, df, key_cols):
    total = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dupes = total - distinct
    print(f"{name} — key {key_cols}: {total} rows, {distinct} distinct, {dupes} duplicate(s)")

check_unique("movies",  movies_raw, ["movieId"])
check_unique("links",   links_raw,  ["movieId"])
check_unique("ratings", ratings_raw, ["userId", "movieId"])

movies — key ['movieId']: 9742 rows, 9742 distinct, 0 duplicate(s)
links — key ['movieId']: 9742 rows, 9742 distinct, 0 duplicate(s)
ratings — key ['userId', 'movieId']: 100836 rows, 100836 distinct, 0 duplicate(s)


`key_cols` is just a list of column names — the columns that, together, are supposed to uniquely identify one row.

For example, `check_unique("ratings", ratings_raw, ["userId", "movieId"])` passes `["userId", "movieId"]` as key_cols. Inside the function, `df.dropDuplicates(key_cols)` uses that list to tell Spark: "only look at the userId and movieId columns when deciding if two rows are duplicates — ignore the other columns (rating, timestamp) entirely." So two rows with the same user+movie combo count as one, even if their ratings/timestamps differ.

## Write Bronze Delta tables with ingestion metadata.
Bronze metadata answers only:
- *Where* did we get the data from?
- *When* did we get the data?

Why calculate `BATCH_ID` outside the function, once, at the top? So all 4 tables written in this one run share the exact same batch_id. If it were calculated inside write_to_bronze, each of the 4 calls would get a slightly different timestamp (a few milliseconds apart) — technically true, but useless for grouping "everything from this run" together later.

`{BRONZE_DIR}/bronze_ratings`. **Note**: this isn't one CSV file — Delta creates a folder containing the actual data files plus a hidden _delta_log folder that records every write ever made to that table. That log folder is literally what "Delta" means, and it's what lets us do time-travel later.

**What this cell does:**

1. Define the directory where Bronze tables will be saved.
2. Define the batch ID by converting the current UTC time into a string.
3. Define a function that writes a DataFrame to Bronze, adding metadata. It takes:
   - `df` — the actual data (e.g. `ratings_raw`)
   - `table_name` — a name we choose for the Bronze table
   - `source_file` — just a text label naming which raw file this data came from,
     stamped onto every row as a column (not a second copy of the data itself;
     useful later if a table ever receives data from more than one source)
4. Inside the function, build `tagged`: the same DataFrame as `df`, with three
   new columns added on top — `batch_id`, `ingested_at` (current timestamp),
   and `source_file`.
5. Write `tagged` to disk in Delta format, using `mode("append")` so each run
   adds new rows instead of overwriting what's already there — this is what
   lets Bronze grow. `.save(...)` is what actually triggers the write.
6. Print a manual confirmation message (row count + batch id) so we can see
   the write succeeded directly in the notebook — separate from Delta's own
   internal transaction log.

In [6]:
from pyspark.sql import functions as F     # Spark's library of column-building functions, used 
from datetime import datetime, timezone    

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
          # datetime.now(timezone.utc) — get the current date and time, specifically in UTC, just for uniformity
          # .strftime("%Y%m%d_%H%M%S") — "string format time": turns that date/time into a plain text string following this pattern:
          #  %Y=4-digit year, %m=month, %d=day, %H=hour, %M=minute, %S=second we turn it into a plain string (not just leave it a date object) because it needs to become 
          # a column value and, later, could be used safely in a file path or folder name, no spaces, no colons.

def write_to_bronze(df, table_name, source_file):           # a function to write a DataFrame to the bronze layer in Delta format, adding metadata columns
    tagged = (                                              # a dataframe that shows the original data plus three new columns: batch_id, ingested_at, and source_file
        df # the DataFrame we want to write
        .withColumn("batch_id", F.lit(BATCH_ID))            # F.lit() creates a column with a constant value, we use it to add the batch ID to every row
        .withColumn("ingested_at", F.current_timestamp())   # F.current_timestamp() creates a column with the current timestamp, we use it to record when the data was ingested
        .withColumn("source_file", F.lit(source_file))      # F.lit() creates a column with a constant value, we use it to record the source file for each row
    )
    tagged.write.format("delta").mode("append").save(f"{BRONZE_DIR}/{table_name}") 
    # This is one instruction built by chaining four steps, each configuring one decision:
    # 1. .write — switch from sitting in memory to write to disk, keeing at in memory is not persistent, it will be lost when the Spark session ends
    # 2. .format("delta") — write in Delta format, which is a special format that allows for ACID transactions and other features
    # 3. .mode("append") — if the table already exists, add to it instead of overwriting it. This what makes Bronze table growable
    # 4. .save(f"{BRONZE_DIR}/{table_name}") — the actual trigger that runs everything and writes files to disk
    print(f"wrote {tagged.count()} rows to {table_name} (batch {BATCH_ID})")       # print how many rows were written to the table, and which batch ID was used

write_to_bronze(ratings_raw, "bronze_ratings", "ratings.csv")
write_to_bronze(movies_raw,  "bronze_movies",  "movies.csv")
write_to_bronze(tags_raw,    "bronze_tags",    "tags.csv")
write_to_bronze(links_raw,   "bronze_links",   "links.csv")

wrote 100836 rows to bronze_ratings (batch 20260917_071842)
wrote 9742 rows to bronze_movies (batch 20260917_071842)
wrote 3683 rows to bronze_tags (batch 20260917_071842)
wrote 9742 rows to bronze_links (batch 20260917_071842)


## Verify Bronze actually grew, by reading it back from disk
`spark.read.format("delta").load(...)` reads the Delta table back from disk, rather than using the ratings_raw etc. DataFrames still sitting in memory from Cell 2. This is deliberate — we want to check what's actually persisted, not just trust what we think we wrote.

`bronze_df.count()` — total rows now sitting in that table.
`bronze_df.select("batch_id").distinct().count()` — how many different batch IDs show up. Since `BATCH_ID` gets recalculated fresh each time Cell 1... actually Cell 5 runs (it's defined right there, not in Cell 1), this should show 2 — proof that two separate ingestion runs are both preserved side-by-side, not merged or overwritten.

In [7]:
# read the bronze tables back in and print the number of rows and batches for each table
# if we rerun the previous cell, the number of rows will increase but the number of batches will stay the same since we are appending to the same table
for name in ["bronze_ratings", "bronze_movies", "bronze_tags", "bronze_links"]:
    bronze_df = spark.read.format("delta").load(f"{BRONZE_DIR}/{name}")
    total = bronze_df.count()
    batches = bronze_df.select("batch_id").distinct().count()
    print(f"{name}: {total} rows across {batches} batch(es)")

bronze_ratings: 403344 rows across 4 batch(es)
bronze_movies: 38968 rows across 4 batch(es)
bronze_tags: 14732 rows across 4 batch(es)
bronze_links: 38968 rows across 4 batch(es)


### Assume in a real-life project you pressed rerun by mistake, what can you do?

You can undo it, and this is exactly why we're using Delta instead of plain CSV/Parquet. Every write to a Delta table creates a new "version," and the full history sticks around. Two ways to fix it:
- Time travel / restore — Delta can literally rewind the table to before the accidental write:
 `spark.sql(f"RESTORE TABLE delta.`{BRONZE_DIR}/bronze_ratings` TO VERSION AS OF 0")`
   This is a real, built-in "undo" — something you simply can't do with plain files (once you've appended to a CSV, the old version is gone unless you separately backed it up).
- Delete just the bad batch — since we tagged every row with batch_id, you can surgically remove only the mistaken run:
  `spark.sql(f"DELETE FROM delta.`{BRONZE_DIR}/bronze_ratings` WHERE batch_id = '<the bad one>'")`
   This is Delta supporting row-level DELETE, which plain Parquet/CSV also can't do without rewriting the whole file yourself.

But here's the more interesting real-world answer: in a well-designed pipeline, you often wouldn't clean it up at all — you'd prevent it from mattering in the first place. Bronze's job is to record everything that was ever received, including accidental reruns — deleting from it kind of defeats the point of an untouched audit log. The actual fix usually lives one layer up: Silver's job is to dedupe/reconcile before anything "counts," so an accidental double-ingest into Bronze doesn't corrupt any real numbers downstream — it's just some wasted storage, not a correctness bug. Whether you bother cleaning Bronze itself becomes a cost/storage decision, not a correctness one.